In [0]:
%sql
drop table if exists banking.metadata.tables;
drop table if exists banking.metadata.table_parameters;
drop table if exists banking.metadata.table_watermarks;
drop table if exists banking.metadata.pipeline_runs;

In [0]:
%sql
create catalog if not exists banking;
create schema if not exists banking.metadata;

-- metadata.tables
-- static registry of logical tables

create table if not exists banking.metadata.tables(
    table_id        int,
    table_name      string,
    source_system   string,     --sqlserver/blob
    source_schema   string,     --dbo(null for blob)
    source_table    string,     --table-name(null for blob)
    source_path     string,     --blob-path(null for sqlserver)
    target_layer    string,     --silver/gold
    bronze_schema   string,     --bronze
    silver_schema   string,     --silver
    gold_schema     string,     --gold
    active_flag     boolean,
    load_order      int,
    created_at      timestamp
)
using delta;

-- metadata.table_parameters
-- Processing configuration (load type, PK, watermark)

create table if not exists banking.metadata.table_parameters(
    table_id        int,
    parameter_name  string,     --load_type/PK/watermark_column
    parameter_value string,
    created_at      timestamp
)
using delta;

-- metadata.table_watermarks
-- stores last successful watermark per table
create table if not exists banking.metadata.table_watermarks(
    table_id                int,
    last_watermark_value    string,     --flexible type storage
    last_updated_at         timestamp,
    last_run_id             bigint
)
using delta
partitioned by (table_id);

--metadata.table_pipeline_runs
--Execution audit table
create table if not exists banking.metadata.pipeline_runs(
    run_id                  bigint,
    table_id                int,
    layer                   string,
    start_time              timestamp,
    end_time                timestamp,
    status                  string,
    number_of_records       bigint,
    error_message           string
)
using delta
partitioned by (table_id);

In [0]:
%sql
-- create source schema
create schema if not exists banking.source;

-- create volume for source files (blob loading)
create volume if not exists banking.source.volume;

In [0]:
%sql
-- INSERT INTO metadata.tables

INSERT INTO banking.metadata.tables VALUES
-- 1. Customers (SQL Server)
(1, 'customers', 'sqlserver', 'banking', 'customers', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 1, current_timestamp()),

-- 2. Accounts (SQL Server)
(2, 'accounts', 'sqlserver', 'banking', 'accounts', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 2, current_timestamp()),

-- 3. Transactions (SQL Server)
(3, 'transactions', 'sqlserver', 'banking', 'transactions', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 3, current_timestamp()),

-- 4. Branches (SQL Server - Full Load)
(4, 'branches', 'sqlserver', 'banking', 'branches', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 4, current_timestamp()),

-- 5. Credit Bureau Reports (Blob CSV)
(5, 'credit_bureau_reports', 'blob', NULL, NULL,
 '/Volumes/banking/source/volume/credit_bureau_reports/', 'silver',
 'bronze', 'silver', NULL, TRUE, 5, current_timestamp()),

-- 6. Payment Gateway Logs (Blob CSV)
(6, 'payment_gateway_logs', 'blob', NULL, NULL,
 '/Volumes/banking/source/volume/payment_gateway_logs/', 'silver',
 'bronze', 'silver', NULL, TRUE, 6, current_timestamp());

In [0]:
%sql
-- INSERT INTO metadata.table_parameters

INSERT INTO banking.metadata.table_parameters VALUES

-- ================= CUSTOMERS =================
(1, 'load_type', 'MERGE', current_timestamp()),
(1, 'primary_key', 'customer_id', current_timestamp()),
(1, 'watermark_column', 'updated_at', current_timestamp()),

-- ================= ACCOUNTS =================
(2, 'load_type', 'MERGE', current_timestamp()),
(2, 'primary_key', 'account_id', current_timestamp()),
(2, 'watermark_column', 'updated_at', current_timestamp()),

-- ================= TRANSACTIONS =================
(3, 'load_type', 'APPEND', current_timestamp()),
(3, 'primary_key', 'txn_id', current_timestamp()),
(3, 'watermark_column', 'txn_timestamp', current_timestamp()),

-- ================= BRANCHES =================
(4, 'load_type', 'FULL', current_timestamp()),
(4, 'primary_key', 'branch_code', current_timestamp()),

-- ================= CREDIT BUREAU REPORTS =================
(5, 'load_type', 'MERGE', current_timestamp()),
(5, 'primary_key', 'customer_id', current_timestamp()),
(5, 'watermark_column', 'bureau_pull_date', current_timestamp()),

-- ================= PAYMENT GATEWAY LOGS =================
(6, 'load_type', 'APPEND', current_timestamp()),
(6, 'primary_key', 'txn_id', current_timestamp()),
(6, 'watermark_column', 'processed_timestamp', current_timestamp());


In [0]:
%sql

-- OPTIONAL: INITIALIZE WATERMARK TABLE
-- (Only for INCREMENTAL tables)

INSERT INTO banking.metadata.table_watermarks VALUES
(1, '1900-01-01 00:00:00', current_timestamp(), NULL),
(2, '1900-01-01 00:00:00', current_timestamp(), NULL),
(3, '1900-01-01 00:00:00', current_timestamp(), NULL),
(5, '1900-01-01 00:00:00', current_timestamp(), NULL),
(6, '1900-01-01 00:00:00', current_timestamp(), NULL);

In [0]:
%sql
INSERT INTO banking.metadata.tables
VALUES (
    7,
    'customer_360',
    'silver',
    NULL,
    NULL,
    NULL,
    'gold',
    NULL,
    NULL,
    'gold',
    TRUE,
    1,
    current_timestamp()
);

-- COMMAND ----------

INSERT INTO banking.metadata.tables
VALUES
(
8,
'branch_performance',
'silver',
NULL,
NULL,
NULL,
'gold',
NULL,
NULL,
'gold',
TRUE,
2,
current_timestamp()
),

(
9,
'transaction_channel_summary',
'silver',
NULL,
NULL,
NULL,
'gold',
NULL,
NULL,
'gold',
TRUE,
3,
current_timestamp()
),

(
10,
'daily_bank_kpi',
'silver',
NULL,
NULL,
NULL,
'gold',
NULL,
NULL,
'gold',
TRUE,
4,
current_timestamp()
);

-- COMMAND ----------

INSERT INTO banking.metadata.tables
VALUES (
    11,
    'risk_customer_summary',
    'silver',
    NULL,
    NULL,
    NULL,
    'gold',
    NULL,
    NULL,
    'gold',
    TRUE,
    1,
    current_timestamp()
);